# Lab 16: DQN on CartPole

**Module 16** | Read `notes/16-dqn.pdf` before running this notebook.

Train a DQN agent with Stable-Baselines3 on CartPole-v1 and plot episode rewards.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


# Lab 16: DQN on CartPole

**Module 16** | Read `notes/16-dqn.pdf` before running this notebook.

Train a DQN agent with Stable-Baselines3 on CartPole-v1 and plot episode rewards.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Deep Q-Networks on CartPole

CartPole-v1 is a classic control benchmark: keep a pole balanced on a cart by pushing left or right.
The state is four continuous values (cart position, velocity, pole angle, angular velocity); the agent chooses one of two discrete actions.

**Deep Q-Networks (DQN)** approximate the action-value function $Q(s,a)$ with a neural network.
Experience replay and a target network stabilize training when bootstrapping from the network's own predictions.
Stable-Baselines3 wraps these details so you can focus on environment setup and evaluation.

In this lab you will:

1. Wrap the environment with `Monitor` so episode rewards are logged automatically.
2. Train a DQN policy for **30,000 environment steps**.
3. Plot the per-episode reward curve and interpret whether the agent is learning.


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor

ROOT = Path("..").resolve()

env = Monitor(gym.make("CartPole-v1"))
model = DQN("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=30_000)

rewards = env.get_episode_rewards()
print(f"Episodes completed: {len(rewards)}")
print(f"Mean reward (last 10): {np.mean(rewards[-10:]):.1f}")
print(f"Max episode reward: {max(rewards)}")

plt.figure(figsize=(8, 4))
plt.plot(rewards, alpha=0.7, label="Episode reward")
if len(rewards) >= 10:
    window = np.convolve(rewards, np.ones(10) / 10, mode="valid")
    plt.plot(range(9, 9 + len(window)), window, linewidth=2, label="10-episode moving avg")
plt.axhline(195, color="gray", linestyle="--", label="solved threshold (195)")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN on CartPole-v1")
plt.legend()
plt.tight_layout()
plt.show()

env.close()


## Interpretation

CartPole is considered **solved** when the agent averages at least **195** reward over 100 consecutive episodes (each step gives +1 until the pole falls, max 500).

A healthy learning curve shows rewards climbing from near-random (~20 to 40) toward the maximum.
If rewards plateau early, try more timesteps or tune hyperparameters (`learning_rate`, `buffer_size`, `exploration_fraction`) in a follow-up experiment.

**Key takeaway:** value-based RL with function approximation can solve simple control tasks without hand-crafted features; the MLP learns representations directly from raw state vectors.
